# Evaluator: Visibility Extent (Updated)

This notebook evaluates `Visibility_extent(20261123-093758).py`. Separate sender and receiver crowds use bounded `knowledge_breadth`, and all outcomes concern the **receiver** crowd.

The main figures show best-solution rank, pairwise solution diversity, and adopted-solution fitness across visibility extent. New figures show **generative potential**, using the same overall-across-K and by-K formats as the existing outcomes. Additional figures examine breakthrough fitness, uniqueness, adoption-quality trajectories, and extent-by-K heatmaps.

**Input:** Seven pickle output families produced by `Visibility_extent.py`, including the new `generative_potential_across_K` family. No simulation rerun is required. The notebook saves **PNG figures only**, not CSV files. No synthetic or actual experiment results are embedded in this notebook.

## 1. Configuration and methods

- Experiment settings: `N=18`, `K=9, ..., 17`, **100 solvers in each crowd**, `knowledge_breadth=12`, `search_iteration=300`, `visibility_interval=10`, and 15 visibility extent conditions.
- The simulation uses **400 independent landscape repetitions**, each with **5 solver resets**. Each saved scalar is already averaged across the repetitions and resets for each `(extent, K)`; these files do not contain run-level observations.
- **Visibility extent** is the proportion of sender solvers whose solutions remain eligible for exposure throughout the run. At extent `0`, no sender solutions are visible (search-apart benchmark).
- Figures averaging across `K` use equal weights across all K values, with `NaN` values omitted for undefined adoption-dependent outcomes.
- Saved means do not retain repetition-level variation; confidence intervals or significance tests cannot be recovered from these pickle files alone.
- `adopted_solution_fitness` is the mean true fitness of adopted solutions conditional on adoption; its time trajectory is cumulative. `generative_potential` evaluates sampled successful adoptions via independent post-adoption continuations. Both are undefined (`NaN`) when there is no qualifying adoption; zero visibility therefore has no adoption-based value.

**Edit `result_folder`** to point to your pickle outputs. Figures are saved in its `visibility_extent_figures` subfolder.

In [ ]:
from pathlib import Path
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Matched to the earlier evaluator notebook.
legend_properties = {'weight': 'bold'}
nus_blue = '#003D7C'
nus_orange = '#EF7C00'
morandi_blue = '#046586'
morandi_green = '#28A9A1'
morandi_yellow = '#C9A77C'
morandi_orange = '#F4A016'
morandi_pink = '#F6BBC6'
morandi_red = '#E71F19'
morandi_purple = '#B08BEB'
shallow_grey = '#D3D4D3'

K_colors = [morandi_blue, morandi_orange, morandi_green, morandi_purple,
            morandi_red, morandi_yellow, '#555555', '#B75F91', morandi_pink]


def safe_nanmean(values, axis=None):
    """Ignore structurally undefined adoption values, retaining all-NaN as NaN."""
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        return np.nanmean(values, axis=axis)


In [ ]:
# =========================
# Experiment configuration — same as Visibility_extent.py
# =========================
N = 18
agent_num = 100
knowledge_breadth = 12
landscape_iteration = 400  # Documentation only; averages were already computed upstream.
solver_reset_num = 5  # Independent crowd restarts per landscape, averaged upstream.
search_iteration = 300
K_list = list(range(9, N))
visibility_extent_list = [0.0, 0.01, 0.02, 0.04, 0.08,
                          0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
visibility_interval = 10

# Change this to the folder containing your new pickle outputs, if auto-detection fails.
result_folder = "/Volumes/T7/data/gst-1010-26/ISR_Visibility_4/Extent"  # e.g., Path('/Volumes/T7/data/your-new-experiment')

strict_loading = True  # Set False to view partial results from an unfinished run.

candidate_folders = [Path.cwd(), Path.cwd() / 'Results', Path.cwd() / 'results',
                     Path.cwd() / 'output', Path.cwd() / 'data']
expected_example = (f'visibility_extent_{visibility_extent_list[0]}_interval_{visibility_interval}'
                    f'_breakthrough_rank_across_K_size_{agent_num}')

if result_folder is None:
    result_folder = next((folder for folder in candidate_folders
                          if (folder / expected_example).is_file()), Path.cwd())
result_folder = Path(result_folder).expanduser()
figure_folder = result_folder / 'visibility_extent_figures'
figure_folder.mkdir(parents=True, exist_ok=True)

print('Result folder:', result_folder)
print('Output folder:', figure_folder)
print('Example expected file:', expected_example)
print('Folder exists:', result_folder.exists())

## 2. Load and validate the seven outcome families

Files follow the simulation's naming convention:

`visibility_extent_{extent}_interval_{interval}_{DV}_across_K_size_{agent_num}`

The **six scalar outcomes** (including `generative_potential`) load as arrays of shape `(extent, K)`. The adoption-fitness trajectory loads as `(extent, K, visibility event)`. The loader checks every expected file and shape, preserves structurally undefined values as `NaN`, and reports missing/invalid files without shifting extent rows.

In [ ]:
scalar_dvs = [
    'breakthrough_rank',
    'breakthrough_fitness',
    'pairwise_diversity',
    'unique_solution_count',
    'adopted_solution_fitness',
    'generative_potential',
]
trajectory_dv = 'adopted_solution_fitness_across_time'
expected_events = search_iteration // visibility_interval


def result_file(extent, dv):
    filename = (f'visibility_extent_{extent}_interval_{visibility_interval}_'
                f'{dv}_across_K_size_{agent_num}')
    return result_folder / filename


def load_pickle(path):
    with open(path, 'rb') as infile:
        return pickle.load(infile)


def load_visibility_extent_results(strict=True):
    n_extent = len(visibility_extent_list)
    n_K = len(K_list)
    data = {dv: np.full((n_extent, n_K), np.nan) for dv in scalar_dvs}
    data[trajectory_dv] = np.full((n_extent, n_K, expected_events), np.nan)
    missing_files = []
    invalid_files = []

    for i, extent in enumerate(visibility_extent_list):
        for dv in scalar_dvs + [trajectory_dv]:
            path = result_file(extent, dv)
            if not path.is_file():
                missing_files.append(path.name)
                continue
            try:
                values = np.asarray(load_pickle(path), dtype=float)
            except (OSError, pickle.UnpicklingError, TypeError, ValueError) as exc:
                invalid_files.append((path.name, str(exc)))
                continue
            shape = (n_K, expected_events) if dv == trajectory_dv else (n_K,)
            if values.shape != shape:
                invalid_files.append((path.name, f'got {values.shape}; expected {shape}'))
                continue
            data[dv][i] = values

    expected_file_count = n_extent * (len(scalar_dvs) + 1)
    print(f'Files loaded: {expected_file_count - len(missing_files) - len(invalid_files)} / {expected_file_count}')
    print('Array shapes:', {dv: value.shape for dv, value in data.items()})

    if missing_files:
        print(f'\nMissing files ({len(missing_files)}):')
        for filename in missing_files[:12]:
            print('  ', filename)
        if len(missing_files) > 12:
            print('  ...')
    if invalid_files:
        print(f'\nInvalid files ({len(invalid_files)}):')
        for name, issue in invalid_files[:12]:
            print('  ', name, ':', issue)

    if strict and (missing_files or invalid_files):
        raise FileNotFoundError(
            'Missing/invalid visibility-extent results. Edit result_folder, agent_num, '
            'visibility_interval, the extent grid, or set strict_loading=False '
            'for incomplete runs. First expected file: ' + str(result_file(visibility_extent_list[0], 'breakthrough_rank'))
        )
    if not any(np.isfinite(data[dv]).any() for dv in scalar_dvs):
        raise FileNotFoundError('No valid scalar pickle outputs found in ' + str(result_folder))
    return data


data = load_visibility_extent_results(strict=strict_loading)

In [ ]:
# =========================
# Data checks and long-form details
# =========================
checks = {
    'breakthrough_rank': (1, 2 ** N),
    'breakthrough_fitness': (0, 1),
    'pairwise_diversity': (0, 1),
    'unique_solution_count': (1, agent_num),
    'adopted_solution_fitness': (0, 1),
}
for dv, (minimum, maximum) in checks.items():
    finite = data[dv][np.isfinite(data[dv])]
    outside = np.count_nonzero((finite < minimum - 1e-8) | (finite > maximum + 1e-8))
    print(f'{dv:29s} finite={finite.size:3d} / {data[dv].size:3d}; outside expected range={outside}')

# Generative potential is a post-adoption continuation measure.
# Do not impose a range unless the measure's normalization is verified.
print('generative_potential'.ljust(30),
      f"finite={np.isfinite(data['generative_potential']).sum():3d} / {data['generative_potential'].size:3d}")

# The trajectory's final cumulative mean should match the scalar adoption-quality
# measure for each (extent, K), up to floating-point differences.
last_trajectory = data[trajectory_dv][:, :, -1]
both_defined = np.isfinite(last_trajectory) & np.isfinite(data['adopted_solution_fitness'])
if both_defined.any():
    gap = np.max(np.abs(last_trajectory[both_defined] - data['adopted_solution_fitness'][both_defined]))
    print('Max discrepancy: final trajectory vs. scalar adoption quality =', round(float(gap), 12))

# No adoption-dependent results should be numerically defined at zero visibility.
zero_index = visibility_extent_list.index(0.0)
for dv in ['adopted_solution_fitness', 'generative_potential']:
    if np.isfinite(data[dv][zero_index]).any():
        warnings.warn(f'{dv} has non-NaN observations at zero visibility; inspect source output.')

rows = []
for i, extent in enumerate(visibility_extent_list):
    for j, K in enumerate(K_list):
        row = {'visibility_extent': extent, 'K': K}
        for dv in scalar_dvs:
            row[dv] = data[dv][i, j]
        rows.append(row)

detailed_results = pd.DataFrame(rows)
print('\nDetailed data preview (one observation per extent × K):')
display(detailed_results.head(12).round(4))

## 3. Main results: visibility extent

The overall curves average each outcome equally across **all K values** in `K_list`. The adjacent by-K figures show heterogeneity across landscape complexity. Lines join the sampled visibility extents in their actual numerical order, including the finer grid close to zero.

In [ ]:
# =========================
# Best solution rank across visibility extent (average across K)
# =========================
fig, ax = plt.subplots(figsize=(7.2, 5.0))

# -------------------------
# Axis formatting
# -------------------------
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color("black")

ax.grid(alpha=0.5)
ax.tick_params(axis="both", labelsize=11)

# -------------------------
# Data
# -------------------------
x = np.asarray(visibility_extent_list)
y_rank = safe_nanmean(data["breakthrough_rank"], axis=1)

# Identify search-apart benchmark point
zero_idx = np.where(np.isclose(x, 0.0))[0][0]
zero_x, zero_y = x[zero_idx], y_rank[zero_idx]

# -------------------------
# Main curve
# -------------------------
ax.plot(
    x,
    y_rank,
    marker="o",
    linewidth=2.5,
    markersize=7,
    color="k",
    markerfacecolor="white",
    markeredgecolor="black",
    markeredgewidth=1.5,
    zorder=2,
)

# Give endpoint observations some breathing room
ax.set_xlim(-0.05, 1.05)
ax.margins(y=0.10)

# -------------------------
# Horizontal benchmark line
# -------------------------
ax.axhline(
    y=zero_y,
    color="gray",
    linestyle="--",
    linewidth=1.5,
    zorder=1,
)

# Highlight search-apart benchmark
ax.scatter(
    zero_x,
    zero_y,
    s=70,
    color="black",
    zorder=3,
)

# -------------------------
# Annotation-box style
# -------------------------
textbox_style = dict(
    boxstyle="round,pad=0.35",
    facecolor="white",
    edgecolor="black",
    linewidth=1.0,
)

arrow_style = dict(
    arrowstyle="->",
    linewidth=1.1,
    color="black",
    shrinkA=5,
    shrinkB=5,
)

# -------------------------
# Search-apart annotation
# -------------------------
ax.annotate(
    "Search-apart",
    xy=(zero_x, zero_y),
    xycoords="data",
    xytext=(0.15, 0.82),
    textcoords="axes fraction",
    ha="left",
    va="center",
    fontsize=12,
    bbox=textbox_style,
    arrowprops=arrow_style,
    annotation_clip=True,
    zorder=4,
)

# -------------------------
# Axes
# -------------------------
ax.invert_yaxis()  # Rank 1 is best; invert so a better rank is visually higher.

ax.set_xlabel(
    "Visibility extent",
    fontweight="bold",
    fontsize=12,
)

ax.set_ylabel(
    "Best solution rank (lower is better)",
    fontweight="bold",
    fontsize=12,
)

fig.tight_layout()

plt.savefig(
    figure_folder / "Breakthrough_across_visibility_extent.png",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)

plt.show()

In [ ]:
# Best solution rank across visibility extent (by K).
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')
ax.grid(alpha=0.5)
ax.tick_params(axis='both', labelsize=11)
for j, K in enumerate(K_list):
    ax.plot(visibility_extent_list, data['breakthrough_rank'][:, j],
            marker='o', linewidth=2.0, markersize=4.5,
            label=f'K={K}', color=K_colors[j % len(K_colors)])
ax.invert_yaxis()
ax.set_xlabel('Visibility extent', fontweight='bold', fontsize=12)
ax.set_ylabel('Best solution rank (lower is better)', fontweight='bold', fontsize=12)
ax.legend(frameon=False, ncol=2, prop={'weight': 'bold', 'size': 10})
fig.tight_layout()
plt.savefig(figure_folder / 'Breakthrough_by_K_across_visibility_extent.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
# =========================
# Pairwise diversity across visibility extent (average across K)
# =========================
fig, ax = plt.subplots(figsize=(7.2, 5.0))

# -------------------------
# Axis formatting
# -------------------------
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color("black")

ax.grid(alpha=0.5)
ax.tick_params(axis="both", labelsize=11)

# -------------------------
# Data
# -------------------------
x = np.asarray(visibility_extent_list)
y = safe_nanmean(data["pairwise_diversity"], axis=1)

# Identify zero-visibility and full-visibility benchmark points
zero_idx = np.where(np.isclose(x, 0.0))[0][0]
full_idx = np.where(np.isclose(x, 1.0))[0][0]

zero_x, zero_y = x[zero_idx], y[zero_idx]
full_x, full_y = x[full_idx], y[full_idx]

# -------------------------
# Main curve
# -------------------------
ax.plot(
    x,
    y,
    marker="o",
    linewidth=2.5,
    markersize=7,
    color="k",
    markerfacecolor="white",
    markeredgecolor="black",
    markeredgewidth=1.5,
    zorder=2,
)

# Give endpoint observations some breathing room
ax.set_xlim(-0.05, 1.05)
ax.margins(y=0.10)

# Highlight the two focal observations
ax.scatter(
    zero_x,
    zero_y,
    s=70,
    color="black",
    zorder=3,
)

ax.scatter(
    full_x,
    full_y,
    s=70,
    color="black",
    zorder=3,
)

# -------------------------
# Annotation-box style
# -------------------------
textbox_style = dict(
    boxstyle="round,pad=0.35",
    facecolor="white",
    edgecolor="black",
    linewidth=1.0,
)

arrow_style = dict(
    arrowstyle="->",
    linewidth=1.1,
    color="black",
    shrinkA=5,
    shrinkB=5,
)

# -------------------------
# Zero visibility annotation
# -------------------------
ax.annotate(
    "Search-apart",
    xy=(zero_x, zero_y),
    xycoords="data",
    xytext=(0.14, 0.82),
    textcoords="axes fraction",
    ha="left",
    va="center",
    fontsize=12,
    bbox=textbox_style,
    arrowprops=arrow_style,
    annotation_clip=True,
    zorder=4,
)

# -------------------------
# Full visibility annotation
# -------------------------
ax.annotate(
    "Full visibility",
    xy=(full_x, full_y),
    xycoords="data",
    xytext=(0.58, 0.62),
    textcoords="axes fraction",
    ha="left",
    va="center",
    fontsize=12,
    bbox=textbox_style,
    arrowprops=arrow_style,
    annotation_clip=True,
    zorder=4,
)

# -------------------------
# Axes
# -------------------------
ax.set_xlabel(
    "Visibility extent",
    fontweight="bold",
    fontsize=12,
)

ax.set_ylabel(
    "Pairwise solution diversity",
    fontweight="bold",
    fontsize=12,
)

plt.tight_layout()

output_png = figure_folder / "Pairwise_diversity_across_visibility_extent.png"

plt.savefig(
    output_png,
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)

plt.show()

print("Saved:", output_png)

In [ ]:
# Pairwise diversity across visibility extent (by K).
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')
ax.grid(alpha=0.5)
ax.tick_params(axis='both', labelsize=11)
for j, K in enumerate(K_list):
    ax.plot(visibility_extent_list, data['pairwise_diversity'][:, j],
            marker='o', linewidth=2.0, markersize=4.5,
            label=f'K={K}', color=K_colors[j % len(K_colors)])
ax.set_xlabel('Visibility extent', fontweight='bold', fontsize=12)
ax.set_ylabel('Pairwise solution diversity', fontweight='bold', fontsize=12)
ax.legend(frameon=False, ncol=2, prop={'weight': 'bold', 'size': 10})
fig.tight_layout()
plt.savefig(figure_folder / 'Pairwise_diversity_by_K_across_visibility_extent.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
# Adopted solution fitness across visibility extent (average across K).
# Undefined outcomes (including v=0) stay NaN and are not shown as zero.
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')
ax.grid(alpha=0.5)
ax.tick_params(axis='both', labelsize=11)
y_adopted = safe_nanmean(data['adopted_solution_fitness'], axis=1)
ax.plot(np.asarray(visibility_extent_list), y_adopted, marker='o', linewidth=2.5, markersize=7, color='k',
        markerfacecolor='white', markeredgecolor='black', markeredgewidth=1.5)
ax.set_xlabel('Visibility extent', fontweight='bold', fontsize=12)
ax.set_ylabel('Adopted solution fitness', fontweight='bold', fontsize=12)
fig.tight_layout()
plt.savefig(figure_folder / 'Adopted_solution_quality_across_visibility_extent.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
# Adopted solution fitness across visibility extent (by K).
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')
ax.grid(alpha=0.5)
ax.tick_params(axis='both', labelsize=11)
for j, K in enumerate(K_list):
    ax.plot(visibility_extent_list, data['adopted_solution_fitness'][:, j],
            marker='o', linewidth=2.0, markersize=4.5,
            label=f'K={K}', color=K_colors[j % len(K_colors)])
ax.set_xlabel('Visibility extent', fontweight='bold', fontsize=12)
ax.set_ylabel('Adopted solution fitness', fontweight='bold', fontsize=12)
ax.legend(frameon=False, ncol=2, prop={'weight': 'bold', 'size': 10})
fig.tight_layout()
plt.savefig(figure_folder / 'Adopted_solution_fitness_by_K_across_visibility_extent.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

## 4. Generative potential

Generative potential is calculated for sampled successful adoption events using independent post-adoption search continuations. The upstream script averages this measure across runs with qualifying adoptions. The figures below compare **visibility extent** while retaining the existing figure sizes, markers, gridlines, and fonts. `NaN` values (particularly at extent `0`) are not replaced with zeros.

In [ ]:
# Generative potential across visibility extent (average across K).
# Undefined outcomes (including extent=0) remain NaN.
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')
ax.grid(alpha=0.5)
ax.tick_params(axis='both', labelsize=11)
y_generative = safe_nanmean(data['generative_potential'], axis=1)
ax.plot(np.asarray(visibility_extent_list), y_generative, marker='o', linewidth=2.5, markersize=7, color='k',
        markerfacecolor='white', markeredgecolor='black', markeredgewidth=1.5)
ax.set_xlabel('Visibility extent', fontweight='bold', fontsize=12)
ax.set_ylabel('Generative potential', fontweight='bold', fontsize=12)
fig.tight_layout()
plt.savefig(figure_folder / 'Generative_potential_across_visibility_extent.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
# Generative potential across visibility extent (by K).
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')
ax.grid(alpha=0.5)
ax.tick_params(axis='both', labelsize=11)
for j, K in enumerate(K_list):
    ax.plot(visibility_extent_list, data['generative_potential'][:, j],
            marker='o', linewidth=2.0, markersize=4.5,
            label=f'K={K}', color=K_colors[j % len(K_colors)])
ax.set_xlabel('Visibility extent', fontweight='bold', fontsize=12)
ax.set_ylabel('Generative potential', fontweight='bold', fontsize=12)
ax.legend(frameon=False, ncol=2, prop={'weight': 'bold', 'size': 10})
fig.tight_layout()
plt.savefig(figure_folder / 'Generative_potential_by_K_across_visibility_extent.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

## 5. Additional outcomes

`breakthrough_fitness` is the highest final receiver-solution fitness within a repetition; `breakthrough_rank` is the best final receiver solution's rank in the landscape. These are related but distinct summary scales. `unique_solution_count` captures the number of distinct final receiver solutions, whereas **pairwise diversity** captures how different those solutions are, even when uniqueness coincides.

In [ ]:
# Best solution fitness across visibility extent (average across K).
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')
ax.grid(alpha=0.5)
ax.tick_params(axis='both', labelsize=11)
y_fitness = safe_nanmean(data['breakthrough_fitness'], axis=1)
ax.plot(np.asarray(visibility_extent_list), y_fitness, marker='o', linewidth=2.5, markersize=7, color='k',
        markerfacecolor='white', markeredgecolor='black', markeredgewidth=1.5)
ax.set_xlabel('Visibility extent', fontweight='bold', fontsize=12)
ax.set_ylabel('Best solution fitness', fontweight='bold', fontsize=12)
fig.tight_layout()
# plt.savefig(figure_folder / 'Best_solution_fitness_across_visibility_extent.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
# Unique solution count across visibility extent (average across K).
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')
ax.grid(alpha=0.5)
ax.tick_params(axis='both', labelsize=11)
y_unique = safe_nanmean(data['unique_solution_count'], axis=1)
ax.plot(np.asarray(visibility_extent_list), y_unique, marker='o', linewidth=2.5, markersize=7, color='k',
        markerfacecolor='white', markeredgecolor='black', markeredgewidth=1.5)
ax.set_xlabel('Visibility extent', fontweight='bold', fontsize=12)
ax.set_ylabel('Number of unique solutions', fontweight='bold', fontsize=12)
fig.tight_layout()
# plt.savefig(figure_folder / 'Unique_solution_count_across_visibility_extent.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

## 6. Adoption-quality trajectories across search time

Set `trajectory_visibility_extent` to any sampled visibility extent. The horizontal axis gives actual search periods (`10, 20, ..., 300` by default). Values are **cumulative mean fitness of adopted solutions through that event**, not fitness of only that period's adoptions or mean fitness of all receiver solutions. The first figure averages all K values; the second plots every K separately.

In [ ]:
trajectory_visibility_extent = 1.0  # Change to any entry in visibility_extent_list.
if trajectory_visibility_extent not in visibility_extent_list:
    raise ValueError('trajectory_visibility_extent must appear in visibility_extent_list.')
trajectory_index = visibility_extent_list.index(trajectory_visibility_extent)
trajectory_periods = (np.arange(1, expected_events + 1) * visibility_interval)
trajectory_by_K = data[trajectory_dv][trajectory_index]  # (K, visibility event)
print('Trajectory extent:', trajectory_visibility_extent)
print('Visibility events:', len(trajectory_periods), '| last search period:', trajectory_periods[-1])
print('Defined trajectory entries:', np.isfinite(trajectory_by_K).sum(), '/', trajectory_by_K.size)

In [ ]:
# Cumulative adopted solution fitness across search periods (average across K).
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')
ax.grid(alpha=0.5)
ax.tick_params(axis='both', labelsize=11)
trajectory_mean = safe_nanmean(trajectory_by_K, axis=0)
ax.plot(trajectory_periods, trajectory_mean, marker='o', linewidth=2.5,
        markersize=6, color='k', markerfacecolor='white', markeredgecolor='black', markeredgewidth=1.5)
ax.set_xlabel('Search period', fontweight='bold', fontsize=12)
ax.set_ylabel('Cumulative adopted solution fitness', fontweight='bold', fontsize=12)
fig.tight_layout()
# plt.savefig(figure_folder / f'Adopted_solution_fitness_across_search_periods_extent_{trajectory_visibility_extent:g}.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
# Cumulative adopted solution fitness across search periods (by K).
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')
ax.grid(alpha=0.5)
ax.tick_params(axis='both', labelsize=11)
for j, K in enumerate(K_list):
    ax.plot(trajectory_periods, trajectory_by_K[j], marker='o',
            linewidth=2.0, markersize=4, color=K_colors[j % len(K_colors)],
            label=f'K={K}')
ax.set_xlabel('Search period', fontweight='bold', fontsize=12)
ax.set_ylabel('Cumulative adopted solution fitness', fontweight='bold', fontsize=12)
ax.legend(frameon=False, ncol=2, prop={'weight': 'bold', 'size': 10})
fig.tight_layout()
# plt.savefig(figure_folder / f'Adopted_solution_fitness_by_K_across_search_periods_extent_{trajectory_visibility_extent:g}.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

## 7. Visibility × landscape complexity

These heatmaps display the underlying saved `(extent, K)` averages without pooling across `K`. `NaN` cells remain blank, especially for adoption quality and generative potential at zero visibility. They help diagnose whether an overall trend is stable or masks heterogeneous responses across landscape complexity.

In [ ]:
# Best solution rank (lower is better) across visibility extent and K.
values = np.ma.masked_invalid(data['breakthrough_rank'].T)  # (K, extent)
fig, ax = plt.subplots(figsize=(9.5, 4.8))
image = ax.imshow(values, aspect='auto', interpolation='nearest', cmap='viridis_r', origin='lower')
ax.set_xticks(np.arange(len(visibility_extent_list)))
ax.set_xticklabels([f'{extent:g}' for extent in visibility_extent_list], rotation=45, ha='right')
ax.set_yticks(np.arange(len(K_list)))
ax.set_yticklabels(K_list)
ax.set_xlabel('Visibility extent', fontweight='bold', fontsize=12)
ax.set_ylabel('K', fontweight='bold', fontsize=12)
ax.set_title('Best solution rank (lower is better)', fontweight='bold', fontsize=12)
fig.colorbar(image, ax=ax, label='Best solution rank (lower is better)', pad=0.02)
fig.tight_layout()
# plt.savefig(figure_folder / 'Best_solution_rank_heatmap_across_visibility_extent_and_K.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()


In [ ]:
# Pairwise solution diversity across visibility extent and K.
values = np.ma.masked_invalid(data['pairwise_diversity'].T)  # (K, extent)
fig, ax = plt.subplots(figsize=(9.5, 4.8))
image = ax.imshow(values, aspect='auto', interpolation='nearest', cmap='viridis', origin='lower')
ax.set_xticks(np.arange(len(visibility_extent_list)))
ax.set_xticklabels([f'{extent:g}' for extent in visibility_extent_list], rotation=45, ha='right')
ax.set_yticks(np.arange(len(K_list)))
ax.set_yticklabels(K_list)
ax.set_xlabel('Visibility extent', fontweight='bold', fontsize=12)
ax.set_ylabel('K', fontweight='bold', fontsize=12)
ax.set_title('Pairwise solution diversity', fontweight='bold', fontsize=12)
fig.colorbar(image, ax=ax, label='Pairwise solution diversity', pad=0.02)
fig.tight_layout()
# plt.savefig(figure_folder / 'Pairwise_diversity_heatmap_across_visibility_extent_and_K.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()


In [ ]:
# Adopted solution fitness across visibility extent and K.
values = np.ma.masked_invalid(data['adopted_solution_fitness'].T)  # (K, extent)
fig, ax = plt.subplots(figsize=(9.5, 4.8))
image = ax.imshow(values, aspect='auto', interpolation='nearest', cmap='viridis', origin='lower')
ax.set_xticks(np.arange(len(visibility_extent_list)))
ax.set_xticklabels([f'{extent:g}' for extent in visibility_extent_list], rotation=45, ha='right')
ax.set_yticks(np.arange(len(K_list)))
ax.set_yticklabels(K_list)
ax.set_xlabel('Visibility extent', fontweight='bold', fontsize=12)
ax.set_ylabel('K', fontweight='bold', fontsize=12)
ax.set_title('Adopted solution fitness', fontweight='bold', fontsize=12)
fig.colorbar(image, ax=ax, label='Adopted solution fitness', pad=0.02)
fig.tight_layout()
# plt.savefig(figure_folder / 'Adopted_solution_fitness_heatmap_across_visibility_extent_and_K.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()


In [ ]:
# Generative potential across visibility extent and K.
values = np.ma.masked_invalid(data['generative_potential'].T)  # (K, extent)
fig, ax = plt.subplots(figsize=(9.5, 4.8))
image = ax.imshow(values, aspect='auto', interpolation='nearest', cmap='viridis', origin='lower')
ax.set_xticks(np.arange(len(visibility_extent_list)))
ax.set_xticklabels([f'{extent:g}' for extent in visibility_extent_list], rotation=45, ha='right')
ax.set_yticks(np.arange(len(K_list)))
ax.set_yticklabels(K_list)
ax.set_xlabel('Visibility extent', fontweight='bold', fontsize=12)
ax.set_ylabel('K', fontweight='bold', fontsize=12)
ax.set_title('Generative potential', fontweight='bold', fontsize=12)
fig.colorbar(image, ax=ax, label='Generative potential', pad=0.02)
fig.tight_layout()
plt.savefig(figure_folder / 'Generative_potential_heatmap_across_visibility_extent_and_K.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()